# notebook_testing_arena

Run the ADA Curator v0.4 synthetic fixture through worker retries, release verification, the training reader, tamper detection, and revocation. The optional CPU cell runs ADA training and evaluates the verified test split. [Implementation report](../docs/V0.4_REPORT.md)

Use a Python 3.10+ Jupyter kernel on Linux and install the release extra from this checkout with `python -m pip install -e '.[release]'`. Start Jupyter from the checkout or its `notebooks` directory.

Each full run creates a fresh local scratch directory. It generates original fixture text and makes no network requests. Run from the first cell to start another arena.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
from concurrent.futures import ThreadPoolExecutor
from hashlib import sha256
from pathlib import Path

cwd = Path.cwd().resolve()
repo = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "examples" / "make_release_fixture.py").is_file()
     and (path / "src" / "ada_datasets" / "release.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Start Jupyter from the ADA-Datasets checkout or its notebooks directory.")
if sys.platform != "linux":
    raise RuntimeError("Run the release arena in a Linux kernel.")

sys.path.insert(0, str(repo / "src"))
try:
    import pyarrow
    import tokenizers
except ImportError as exc:
    raise RuntimeError("Install the ADA-Datasets release extra in this kernel's environment.") from exc

from ada_datasets.training import TrainingRelease

arena = Path(tempfile.mkdtemp(prefix="ada-notebook-testing-arena-"))
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(filter(None, (str(repo / "src"), env.get("PYTHONPATH"))))
checks = {}


def run_python(*args, cwd=repo, env=env, timeout=120, check=True):
    result = subprocess.run(
        [sys.executable, *map(str, args)], cwd=cwd, env=env,
        capture_output=True, text=True, timeout=timeout,
    )
    if check and result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result


def curate(*args):
    return json.loads(run_python("-m", "ada_datasets.cli", *args).stdout)


def file_sha(path):
    return sha256(path.read_bytes()).hexdigest()


print(f"Arena directory {arena}")
print(f"PyArrow {pyarrow.__version__} and tokenizers {tokenizers.__version__}")


## Build, retry, and verify

Dispatch the fixture shards as standalone worker processes. Retry one completed task and compare its receipt. Assemble the release using the saved receipts and compare it with a serial build.

In [ ]:
fixture = arena / "fixture"
work = arena / "work"
release_dir = arena / "release"
status = arena / "status.json"
run_python(repo / "examples" / "make_release_fixture.py", "--output-dir", fixture)
recipe = fixture / "recipe.json"
curate("governance", "--status", status, "init")

release_args = ("release", "--recipe", recipe, "--work-dir", work, "--output", release_dir)
prepared = curate(*release_args, "--prepare-only")
assert prepared["prepared"] and prepared["tasks"] >= 2

with ThreadPoolExecutor(max_workers=2) as executor:
    jobs = [
        executor.submit(curate, "worker", "--work-dir", work, "--task-id", task_id)
        for task_id in range(prepared["tasks"])
    ]
    receipts = [job.result() for job in jobs]

assert curate("worker", "--work-dir", work, "--task-id", 0) == receipts[0]
published = curate(*release_args, "--workers", 0, "--status", status)
manifest_sha = published["manifest_sha256"]
verified = curate("verify", "--output", release_dir, "--expected-manifest-sha256", manifest_sha)
assert verified["valid"] and verified["docs"] > 0

serial = curate(
    "release", "--recipe", recipe, "--work-dir", arena / "serial-work",
    "--output", arena / "serial-release", "--workers", 1,
)
assert serial["manifest_sha256"] == manifest_sha
checks["Worker retry and deterministic release"] = "PASS"

manifest = json.loads((release_dir / "manifest.json").read_bytes())
print(f"Verified manifest {manifest_sha}")
print("Split         Documents    Blocks    Active targets")
for split in ("train", "validation", "test"):
    part = manifest["splits"][split]
    assert set(manifest["composition"]["splits"][split]["language"]) == {"en", "ta", "hi"}
    print(f"{split:<14}{part['document_count']:>9}{part['block_count']:>10}{part['active_targets']:>18}")


## Check the training reader and audit

Compare uninterrupted reads with reads resumed from a serialized row-group cursor. Then append bytes to a disposable copy of a token shard and verify that the manifest check rejects it.

In [ ]:
with TrainingRelease(release_dir, manifest_sha, status_path=status) as reader:
    start = reader.cursor("train")
    first, cursor = reader.next_batch("train", start, 1)
    resumed, _ = reader.next_batch("train", json.loads(json.dumps(cursor)), 1)
    contiguous, _ = reader.next_batch("train", start, 2)
    assert first + resumed == contiguous
    assert all(len(row["input_ids"]) == reader.sequence_length + 1 for row in contiguous)
    assert all(len(row["loss_mask"]) == reader.sequence_length for row in contiguous)
checks["Packed targets and cursor resume"] = "PASS"

audit = (release_dir / "audit.jsonl").read_text(encoding="utf-8")
reasons = manifest["totals"]["rejection_reasons"]
assert reasons["duplicate_exact"] > 0 and reasons["duplicate_near"] > 0
assert "fictional-contact@example.invalid" not in audit
checks["Cross-shard deduplication and content-free audit"] = "PASS"

tampered = shutil.copytree(release_dir, arena / "tampered-release")
shard = tampered / manifest["splits"]["train"]["blocks"][0]
with shard.open("ab") as stream:
    stream.write(b"arena tamper check")
rejected = run_python(
    "-m", "ada_datasets.cli", "verify", "--output", tampered,
    "--expected-manifest-sha256", manifest_sha, check=False,
)
assert rejected.returncode != 0 and "artifact digest mismatch" in rejected.stderr
checks["Tampered shard rejected"] = "PASS"
print("Reader, audit, and tamper checks passed.")


## Optional ADA CPU smoke test

For this cell, check out the companion [ADA trainer PR](https://github.com/vimalspaceton618-afk/ADA/pull/13) in a sibling directory called `ADA` and install its `pretraining` extra into the same kernel. Set `ADA_ARENA_CPU_TRAINING=1` before starting Jupyter or set `run_ada_smoke = True` in the cell. Run it before the revocation cell.

In [ ]:
run_ada_smoke = os.environ.get("ADA_ARENA_CPU_TRAINING") == "1"
ada_checkout = repo.parent / "ADA"

if run_ada_smoke:
    trainer = ada_checkout / "scripts" / "ada_pretrain_fineweb_edu.py"
    if not trainer.is_file():
        raise RuntimeError("Set ada_checkout to the companion ADA trainer PR checkout.")
    trainer_env = {**env, "PYTHONPATH": os.pathsep.join((str(ada_checkout), env["PYTHONPATH"]))}
    checkpoints = arena / "checkpoints"
    trainer_args = (
        trainer, "--device", "cpu", "--model-profile", "smoke",
        "--curated-release", release_dir, "--curated-release-sha256", manifest_sha,
        "--release-status-registry", status, "--output-dir", checkpoints,
        "--max-steps", 2, "--batch-size", 1, "--save-every", 1,
        "--eval-every", 1, "--eval-batches", 1,
    )
    run_python(*trainer_args, cwd=ada_checkout, env=trainer_env, timeout=300)
    run_python(*trainer_args, "--resume", "--evaluate-test",
               cwd=ada_checkout, env=trainer_env, timeout=300)
    result = json.loads((checkpoints / "test_evaluation.json").read_bytes())
    assert result["release_manifest_sha256"] == manifest_sha
    assert result["active_targets"] == manifest["splits"]["test"]["active_targets"]
    assert math.isfinite(result["test_loss"])
    checks["ADA CPU train and held-out evaluation"] = "PASS"
    print(f"Evaluated {result['active_targets']} held-out targets from the checkpoint.")
else:
    checks["ADA CPU train and held-out evaluation"] = "OPTIONAL · SKIPPED"
    print(checks["ADA CPU train and held-out evaluation"])


## Verify a successor release

Withdraw the fixture's exact-duplicate alias. Check that the previous reader is quarantined, the impact report names affected shards, and the registry admits the verified replacement.

In [ ]:
successor_recipe = fixture / "successor-recipe.json"
updated = json.loads(recipe.read_bytes())
withdrawn = fixture / "arena-denylist.jsonl"
old_denylist = fixture / updated["denylist"]["path"]
withdrawal = json.dumps({"doc_id": "ada-synthetic:en-train-exact"}).encode()
withdrawn.write_bytes(old_denylist.read_bytes().rstrip(b"\n") + b"\n" + withdrawal + b"\n")
updated["denylist"] = {"path": withdrawn.name, "sha256": file_sha(withdrawn)}
successor_recipe.write_text(json.dumps(updated), encoding="utf-8")
successor = arena / "successor"

with TrainingRelease(release_dir, manifest_sha, status_path=status) as parent:
    replacement = curate(
        "release", "--recipe", successor_recipe, "--work-dir", arena / "successor-work",
        "--output", successor, "--parent", release_dir, "--status", status,
    )
    try:
        parent.next_batch("train", parent.cursor("train"), 1)
    except ValueError as exc:
        assert "quarantined" in str(exc)
    else:
        raise AssertionError("Quarantined parent was still readable")

replacement_sha = replacement["manifest_sha256"]
impact = json.loads((successor / "revocation-impact.json").read_bytes())
assert impact["matched_documents"] == 1 and impact["document_artifacts"] and impact["block_artifacts"]
assert file_sha(release_dir / "manifest.json") == manifest_sha
assert curate("governance", "--status", status, "verify")["active_releases"] == [replacement_sha]
with TrainingRelease(successor, replacement_sha, status_path=status) as current:
    current.next_batch("train", current.cursor("train"), 1)
checks["Revocation, impact report, and successor"] = "PASS"

for check_name, outcome in checks.items():
    print(f"{outcome:<20} {check_name}")


The arena exercises software behavior on one Linux host with synthetic inputs. The real-data GPU pilot and multi-host storage validation are tracked separately. [Pilot approval packet](../docs/WIKIPEDIA_PILOT.md) · [Private worker runbook](../docs/RELEASE_OPERATIONS.md)